# Gradient Descent

This notebook accompanies the **ML Viz** lesson on Gradient Descent.

We start by reproducing the lesson's hand-worked example — fitting a line with mean
squared error (MSE) — then build SGD, Momentum, RMSprop, and Adam from scratch and
visualize their trajectories on a 2D loss surface.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/neural-networks/02-gradient-descent

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — learning is rolling downhill

Training a model *is* minimizing a loss, and **gradient descent** is the workhorse that does it:
compute the gradient (the direction of steepest increase), take a small step the *opposite* way,
repeat. The **learning rate** sets the step size — too small and training crawls, too large and it
overshoots or diverges. On the curved, ill-conditioned loss surfaces of real models, plain descent
zig-zags, so **momentum** and **adaptive** optimizers (RMSprop, Adam) accelerate it. This notebook
fits a line by hand, races the optimizers on a 2-D bowl, and — in the exercises — builds a scalar
**autograd engine** from scratch, exactly the machinery frameworks use.

## Worked example: fitting a line with MSE

This reproduces the lesson's by-hand derivation. The model is a line
$\hat{y} = w x + b$ and the loss is the mean squared error

$$L(w, b) = \frac{1}{n}\sum_{i=1}^{n}\bigl(w x_i + b - y_i\bigr)^2.$$

The gradients (derived via the chain rule on each residual $r_i = w x_i + b - y_i$) are

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_i (w x_i + b - y_i)\,x_i,
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum_i (w x_i + b - y_i).$$

We fit three points that lie exactly on $y = 2x + 1$, starting from $w = b = 0$
with learning rate $\eta = 0.1$ — so the answer should converge to $(w, b) = (2, 1)$.

In [ ]:
# Training data: three points on the line y = 2x + 1
X = np.array([1.0, 2.0, 3.0])
Y = np.array([3.0, 5.0, 7.0])


def mse_loss(w, b):
    """Mean squared error of the line y = w*x + b on (X, Y)."""
    residuals = w * X + b - Y
    return np.mean(residuals ** 2)


def mse_grads(w, b):
    """Analytical gradients dL/dw and dL/db."""
    residuals = w * X + b - Y          # r_i = w*x_i + b - y_i
    dw = 2.0 * np.mean(residuals * X)  # (2/n) * sum(r_i * x_i)
    db = 2.0 * np.mean(residuals)      # (2/n) * sum(r_i)
    return dw, db


def fit_line(w0=0.0, b0=0.0, lr=0.1, n_steps=50):
    """Run gradient descent, recording the (w, b) path and the loss curve."""
    w, b = w0, b0
    ws, bs, losses = [w], [b], [mse_loss(w, b)]
    for _ in range(n_steps):
        dw, db = mse_grads(w, b)
        w = w - lr * dw
        b = b - lr * db
        ws.append(w); bs.append(b); losses.append(mse_loss(w, b))
    return np.array(ws), np.array(bs), np.array(losses)


# Reproduce the lesson's first two hand-computed iterations.
w, b = 0.0, 0.0
print("start: w={:.3f} b={:.3f} L={:.4f}".format(w, b, mse_loss(w, b)))
for step in range(1, 3):
    dw, db = mse_grads(w, b)
    print("  iter {}: dL/dw={:.4f}  dL/db={:.4f}".format(step, dw, db))
    w, b = w - 0.1 * dw, b - 0.1 * db
    print("  iter {}: w={:.3f} b={:.3f} L={:.4f}".format(step, w, b, mse_loss(w, b)))

# Expected:
#   start : w=0.000 b=0.000 L=27.6667
#   iter 1: dL/dw=-22.6667  dL/db=-10.0000  ->  w=2.267 b=1.000 L=0.3319
#   iter 2: dL/dw=2.4889    dL/db=1.0667    ->  w=2.018 b=0.893 L=0.0053

In [ ]:
# Run the full fit and plot (1) the loss curve and (2) the path through (w, b) space.
ws, bs, losses = fit_line(lr=0.1, n_steps=50)
print("converged to w={:.4f}  b={:.4f}  L={:.2e}".format(ws[-1], bs[-1], losses[-1]))

# Loss surface over (w, b) for the contour background.
wg = np.linspace(-1, 4, 200)
bg = np.linspace(-2, 3, 200)
WG, BG = np.meshgrid(wg, bg)
LG = np.zeros_like(WG)
for i in range(WG.shape[0]):
    for j in range(WG.shape[1]):
        LG[i, j] = mse_loss(WG[i, j], BG[i, j])

fig, (ax_loss, ax_path) = plt.subplots(1, 2, figsize=(13, 5))

# (1) Loss curve
ax_loss.semilogy(losses, color='#818cf8', linewidth=2, marker='o', markersize=3)
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('MSE loss (log scale)')
ax_loss.set_title('Loss curve (eta = 0.1)', color='white')

# (2) Parameter path over the loss surface
cs = ax_path.contourf(WG, BG, LG, levels=30, cmap='YlOrRd', alpha=0.7)
plt.colorbar(cs, ax=ax_path, label='Loss')
ax_path.plot(ws, bs, '-o', color='#14b8a6', markersize=3, linewidth=1.6, label='GD path')
ax_path.scatter([2], [1], s=120, color='white', zorder=10, label='True (w, b)=(2, 1)')
ax_path.set_xlabel('w'); ax_path.set_ylabel('b')
ax_path.set_title('Parameter path', color='white')
ax_path.legend()

plt.tight_layout(); plt.show()

**What to notice:** the loss curve plunges then flattens as the `(w, b)` path spirals into the
minimum at `(2, 1)` — the true line `y = 2x + 1`. The first step is huge (steep gradient), later
steps tiny (near-flat) — descent naturally slows as it converges, no schedule required.

### The learning rate, concretely

Same line-fit problem, different step sizes. Edit the `lrs` list and re-run to
explore. Expect: `0.001` crawls, `0.1` converges smoothly, and `0.5` overshoots
and diverges to NaN (it explodes off the chart).

In [ ]:
# Compare learning rates on the line-fit MSE problem.
lrs = [0.001, 0.1, 0.5]
lr_colors = ['#64748b', '#14b8a6', '#f97316']

fig, ax = plt.subplots(figsize=(8, 5))
for lr, color in zip(lrs, lr_colors):
    _, _, losses = fit_line(lr=lr, n_steps=50)
    final = losses[-1]
    final_str = "{:.4g}".format(final) if np.isfinite(final) else "diverged"
    # Mask non-finite values so the divergent curve doesn't break the log plot.
    finite = np.where(np.isfinite(losses), losses, np.nan)
    ax.semilogy(finite, color=color, linewidth=2,
                label="lr={}  ->  L={}".format(lr, final_str))

ax.set_xlabel('Step'); ax.set_ylabel('MSE loss (log scale)')
ax.set_title('Effect of learning rate', color='white')
ax.legend()
plt.tight_layout(); plt.show()

**What to notice:** the learning rate is the whole ballgame. Too **small** and the loss barely
moves in 50 steps; too **large** and it overshoots or oscillates; the middle rate converges fast and
clean. There's no universal best value — it depends on the curvature of the surface.

## Optimizers on a harder surface

The line-fit bowl above is easy — gradient descent walks straight in. Real loss
surfaces have **ravines** (steep one way, flat the other) where plain SGD
struggles. To compare optimizers we switch to a 2D quadratic with asymmetric
curvature:

$$L(w_1, w_2) = 0.4 w_1^2 + 0.9 w_2^2 + 0.1 w_1 w_2$$

The global minimum is at $(0, 0)$. The different eigenvalues of the curvature
make it a good stress test for SGD, Momentum, RMSprop, and Adam.

In [ ]:
def loss(w):
    """2D quadratic bowl with asymmetric curvature."""
    w1, w2 = w
    return 0.4 * w1**2 + 0.9 * w2**2 + 0.1 * w1 * w2

def grad(w):
    """Analytical gradient of loss."""
    w1, w2 = w
    return np.array([0.8 * w1 + 0.1 * w2,
                     1.8 * w2 + 0.1 * w1])

# Plot the surface
w1s = np.linspace(-4, 4, 200)
w2s = np.linspace(-4, 4, 200)
W1, W2 = np.meshgrid(w1s, w2s)
Z = 0.4 * W1**2 + 0.9 * W2**2 + 0.1 * W1 * W2

fig, ax = plt.subplots(figsize=(7, 6))
cs = ax.contourf(W1, W2, Z, levels=25, cmap='YlOrRd', alpha=0.75)
plt.colorbar(cs, ax=ax, label='Loss')
ax.set_xlabel('w₁'); ax.set_ylabel('w₂')
ax.set_title('Loss Surface L(w₁, w₂)', color='white')
plt.tight_layout(); plt.show()

## Implementing optimizers from scratch

Each optimizer is a function `step(w, g, state) → (w_new, state_new)`.

In [ ]:
def sgd_step(w, g, state, lr=0.1):
    """Vanilla SGD: w ← w - lr * g"""
    return w - lr * g, state


def momentum_step(w, g, state, lr=0.1, beta=0.9):
    """SGD with Momentum: accumulates velocity."""
    v = state.get('v', np.zeros_like(w))
    v = beta * v + (1 - beta) * g
    return w - lr * v, {'v': v}


def rmsprop_step(w, g, state, lr=0.05, beta=0.9, eps=1e-8):
    """RMSprop: adaptive learning rates via squared gradient EMA."""
    s = state.get('s', np.zeros_like(w))
    s = beta * s + (1 - beta) * g**2
    return w - lr * g / (np.sqrt(s) + eps), {'s': s}


def adam_step(w, g, state, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8):
    """Adam: combines momentum and RMSprop with bias correction."""
    t = state.get('t', 0) + 1
    m = state.get('m', np.zeros_like(w))
    v = state.get('v', np.zeros_like(w))
    m = beta1 * m + (1 - beta1) * g
    v = beta2 * v + (1 - beta2) * g**2
    m_hat = m / (1 - beta1**t)   # bias correction
    v_hat = v / (1 - beta2**t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps), {'t': t, 'm': m, 'v': v}


def run_optimizer(step_fn, w0, n_steps=60, **kwargs):
    w, state = np.array(w0, dtype=float), {}
    path = [w.copy()]
    losses = [loss(w)]
    for _ in range(n_steps):
        g = grad(w)
        w, state = step_fn(w, g, state, **kwargs)
        path.append(w.copy())
        losses.append(loss(w))
    return np.array(path), losses

In [ ]:
w0 = [3.5, 3.0]
results = {
    'SGD':      run_optimizer(sgd_step,      w0, lr=0.15),
    'Momentum': run_optimizer(momentum_step, w0, lr=0.1),
    'RMSprop':  run_optimizer(rmsprop_step,  w0, lr=0.12),
    'Adam':     run_optimizer(adam_step,     w0, lr=0.3),
}
colors = {'SGD': '#f97316', 'Momentum': '#818cf8', 'RMSprop': '#eab308', 'Adam': '#14b8a6'}

fig, (ax_path, ax_loss) = plt.subplots(1, 2, figsize=(14, 5.5))

# Trajectory plot
ax_path.contourf(W1, W2, Z, levels=25, cmap='Greys', alpha=0.5)
for name, (path, _) in results.items():
    ax_path.plot(path[:, 0], path[:, 1], '-o', markersize=3,
                 color=colors[name], label=name, linewidth=1.8)
ax_path.scatter(0, 0, s=120, color='white', zorder=10, label='Minimum')
ax_path.set_title('Optimization Trajectories', color='white')
ax_path.legend()
ax_path.set_xlim(-4, 4); ax_path.set_ylim(-4, 4)

# Loss curves
for name, (_, losses) in results.items():
    ax_loss.semilogy(losses, color=colors[name], label=name, linewidth=2)
ax_loss.set_xlabel('Step'); ax_loss.set_ylabel('Loss (log scale)')
ax_loss.set_title('Convergence', color='white')
ax_loss.legend()

plt.tight_layout(); plt.show()

**What to notice:** on the asymmetric bowl, **SGD** zig-zags across the steep axis while
crawling along the shallow one; **Momentum** damps the zig-zag; **RMSprop** and **Adam** rescale
per-axis and reach the minimum fastest. The ranking SGD < Momentum < adaptive is why Adam is the
default — the same lesson as the optimization course, now on a neural-net loss.

## Effect of learning rate

Too small = slow convergence. Too large = divergence.

In [ ]:
lrs = [0.01, 0.1, 0.5, 0.9]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, lr in zip(axes, lrs):
    path, losses = run_optimizer(sgd_step, w0, n_steps=80, lr=lr)
    ax.contourf(W1, W2, Z, levels=20, cmap='Greys', alpha=0.5)
    ax.plot(path[:, 0], path[:, 1], '-o', markersize=2.5, color='#818cf8', linewidth=1.5)
    ax.scatter(0, 0, s=80, color='#14b8a6', zorder=10)
    final = losses[-1]
    ax.set_title("lr={}  ->  L={:.4f}".format(lr, final), color='white', fontsize=10)
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)

plt.suptitle('SGD with Different Learning Rates', color='white', y=1.02)
plt.tight_layout(); plt.show()

**What to notice:** sweeping the learning rate shows the failure modes directly — the largest
rate **diverges** (loss explodes), the smallest barely moves, and a well-chosen rate converges
quickly. This is the single most important hyperparameter to get right.

## Momentum and the optimizer family

Plain SGD can crawl through ravines. **Momentum** accumulates a velocity that smooths the path; **Adam** adds per-parameter adaptive step sizes.

In [ ]:
def gd(grad, x0, lr=0.1, steps=50, momentum=0.0):
    x, v, path = x0, 0.0, [x0]
    for _ in range(steps):
        v = momentum * v - lr * grad(x)
        x = x + v
        path.append(x)
    return np.array(path)

# Minimize f(x) = x^2  (grad = 2x)
grad = lambda x: 2 * x
for m in [0.0, 0.9]:
    p = gd(grad, 5.0, lr=0.1, momentum=m)
    print("momentum={}: reached {:.4f} in {} steps".format(m, p[-1], len(p) - 1))

## The library way — validate the line fit against `sklearn`

Our hand-rolled gradient descent should recover the same line a library does in closed form.
`sklearn.LinearRegression` solves the normal equations exactly; the cell checks our GD solution
matches it (and the true `y = 2x + 1`).

In [ ]:
from sklearn.linear_model import LinearRegression

ws, bs, losses = fit_line(lr=0.1, n_steps=300)
w_gd, b_gd = ws[-1], bs[-1]

lin = LinearRegression().fit(X.reshape(-1, 1), Y)
w_sk, b_sk = lin.coef_[0], lin.intercept_

print(f'gradient descent : w = {w_gd:.4f}, b = {b_gd:.4f}')
print(f'sklearn (closed) : w = {w_sk:.4f}, b = {b_sk:.4f}')
assert np.allclose([w_gd, b_gd], [w_sk, b_sk], atol=1e-2), "GD must match sklearn"
assert np.allclose([w_sk, b_sk], [2.0, 1.0], atol=1e-9), "true line is y = 2x + 1"
print('\ngradient descent converges to the exact least-squares solution ✓')

**What to notice:** our iterative gradient descent lands on the *same* `(2, 1)` that
`sklearn`'s closed-form solver finds. For a convex problem like linear regression, GD is just a
slower route to the exact answer — but it's the *only* route once the model is a deep network with
no closed form.

## Gotchas & tradeoffs

- **Learning rate divergence.** Above a curvature-dependent threshold (`2/λ_max`), each step
  overshoots further and the loss explodes — no amount of patience helps.
- **Local minima & saddles.** Convex losses (linear/logistic) have one minimum; neural-net losses
  have many, plus saddle points where the gradient is ~0 but it isn't a minimum.
- **Batch size trades noise for speed.** Full-batch GD is smooth but slow per step; **stochastic**
  (one example) is noisy but fast; **mini-batch** is the practical middle (and the noise can even
  help escape saddles).
- **Adaptive optimizers cost memory** (extra state per parameter) and can generalize slightly
  worse than tuned SGD+momentum.

In [ ]:
# Learning rate too high on the 2-D bowl -> divergence (threshold ~ 2/lambda_max)
for lr in [0.15, 1.0, 2.5]:
    w = np.array([3.5, 3.0])
    diverged = False
    for _ in range(80):
        w = w - lr * grad(w)
        if not np.all(np.isfinite(w)) or np.max(np.abs(w)) > 1e6:
            diverged = True; break
    print(f'lr={lr:>4}: {"DIVERGED" if diverged else f"loss={loss(w):.5f}"}')

**What to notice:** three regimes in one sweep. `lr=0.15` **converges** (loss → 0); `lr=1.0`
is **marginally stable** — it oscillates without shrinking, so the loss stays stuck near its
starting value; and `lr=2.5`, past the stability limit `2/λ_max ≈ 1.1` (the bowl's sharpest
curvature is `λ_max≈1.8`), **diverges**. The safe step size is dictated by the *sharpest*
direction of the loss surface.

## Key takeaways

- Gradient descent steps **downhill**: $w \leftarrow w - \eta\,\nabla L$.
- The **learning rate** $\eta$ trades speed for stability — too big diverges, too small crawls.
- **Momentum** accelerates along consistent directions and damps oscillation.
- **Adam** (adaptive + momentum) is the safe default for deep learning.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — MSE gradients for a line fit

For the model $\hat{y} = wx + b$ with loss $L = \frac{1}{n}\sum_i (\hat{y}_i - y_i)^2$, the chain rule gives

$$\frac{\partial L}{\partial w} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\,x_i, \qquad
\frac{\partial L}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

Implement both. The checks verify the gradients vanish at a perfect fit and match finite differences elsewhere — the exact pair of partial derivatives the worked example above stepped through.

In [ ]:
def mse_gradients(x, y, w, b):
    """Return (dL/dw, dL/db) for the MSE of the line w*x + b."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    # TODO(you): the residuals (predictions minus targets)
    err = ...

    # TODO(you): dL/dw = 2 * mean(err * x)
    dw = ...

    # TODO(you): dL/db = 2 * mean(err)
    db = ...

    return dw, db

In [ ]:
# Checks — run me
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 3.0, 5.0, 7.0])   # exactly y = 2x + 1

dw, db = mse_gradients(x, y, 2.0, 1.0)
assert abs(dw) < 1e-12 and abs(db) < 1e-12, "at the perfect fit, both gradients vanish"

def mse(x, y, w, b):
    return np.mean((w * x + b - y) ** 2)

w0, b0, h = 0.5, -0.3, 1e-6
dw, db = mse_gradients(x, y, w0, b0)
num_dw = (mse(x, y, w0 + h, b0) - mse(x, y, w0 - h, b0)) / (2 * h)
num_db = (mse(x, y, w0, b0 + h) - mse(x, y, w0, b0 - h)) / (2 * h)
assert abs(dw - num_dw) < 1e-5, "dL/dw must match the numerical gradient"
assert abs(db - num_db) < 1e-5, "dL/db must match the numerical gradient"

# Edge case: a single data point -- the gradient still has to reduce cleanly
# (mean over one element is just that element).
dw1, db1 = mse_gradients(np.array([2.0]), np.array([1.0]), 3.0, 0.0)
assert abs(dw1 - 20.0) < 1e-12 and abs(db1 - 10.0) < 1e-12, \
    "n=1: err = 3*2-1 = 5 -> dw = 2*err*x = 20, db = 2*err = 10"

# Edge case: x is all zeros -- dw must vanish regardless of the residual size
# (the weight has no influence on the loss when every feature is 0).
dw0, db0 = mse_gradients(np.array([0.0, 0.0, 0.0]), np.array([5.0, 5.0, 5.0]), 1.0, 1.0)
assert abs(dw0) < 1e-12 and abs(db0 - (-8.0)) < 1e-12, "x=0 -> dw=0, but db still sees the residual"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def mse_gradients(x, y, w, b):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    err = w * x + b - y
    dw = 2 * np.mean(err * x)
    db = 2 * np.mean(err)
    return dw, db
```

</details>

### Exercise 2 — One momentum step

Momentum keeps a running **velocity** that accumulates gradients, then steps along it:

$$v \leftarrow \beta v + \nabla L(w), \qquad w \leftarrow w - \eta\, v$$

Implement a single update. The checks confirm the two facts that matter: $\beta = 0$ reduces to plain gradient descent, and a repeated gradient makes the velocity *grow* ($1 + \beta + \beta^2 + \dots$) — that's the "heavy ball" picking up speed down a consistent slope.

In [ ]:
def momentum_step(w, v, grad, lr=0.1, beta=0.9):
    """One momentum update. Returns the new (w, v)."""
    # TODO(you): update the velocity: beta * v + grad
    v = ...

    # TODO(you): step the weight against the velocity: w - lr * v
    w = ...

    return w, v

In [ ]:
# Checks — run me
w, v = momentum_step(1.0, 0.0, 0.5, lr=0.1, beta=0.0)
assert abs(w - 0.95) < 1e-12 and abs(v - 0.5) < 1e-12, "beta = 0 reduces to plain gradient descent"

_, v = momentum_step(*momentum_step(0.0, 0.0, 1.0, lr=0.1, beta=0.9), 1.0, lr=0.1, beta=0.9)
assert abs(v - 1.9) < 1e-12, "same gradient twice: v = 1 + 0.9 = 1.9 — velocity accumulates"

w, v = 10.0, 0.0
for _ in range(400):
    w, v = momentum_step(w, v, 2 * w, lr=0.05, beta=0.9)   # f(w) = w², grad = 2w
assert abs(w) < 1e-6, "momentum still converges on f(w) = w²"

# Edge case: beta = 1 (no friction at all) -- velocity is just the raw running
# sum of gradients, exactly like an un-damped heavy ball.
w1, v1 = 0.0, 0.0
for _ in range(5):
    w1, v1 = momentum_step(w1, v1, 1.0, lr=0.1, beta=1.0)
assert abs(v1 - 5.0) < 1e-12, "beta=1: v accumulates the gradient with zero decay -> v = 5*1"

# Edge case: zero gradient -- with beta < 1 the velocity decays geometrically
# toward 0 rather than snapping to 0 immediately (momentum has "memory").
w2, v2 = 0.0, 1.0
w2, v2 = momentum_step(w2, v2, 0.0, lr=0.1, beta=0.9)
assert abs(v2 - 0.9) < 1e-12, "grad=0: v carries over as beta*v, it doesn't reset to 0"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def momentum_step(w, v, grad, lr=0.1, beta=0.9):
    v = beta * v + grad
    w = w - lr * v
    return w, v
```

</details>

---
## 🔬 Extra practice — from Open-Deep-ML

Four more drills that build directly on the optimizers above: training a
sigmoid neuron end-to-end with backprop, a tiny scalar autograd engine (a nice
complement to the by-hand chain rule from the worked example), a single
function implementing all three gradient-descent variants, and Adam in its
"optimize this function" signature (a different framing than the `adam_step`
state-update function above).

- [`25` single-neuron-with-backpropagation](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/25_single-neuron-with-backpropagation)
- [`26` implementing-basic-autograd-operations](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/26_implementing-basic-autograd-operations)
- [`47` implement-gradient-descent-variants-with-mse-loss](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/47_implement-gradient-descent-variants-with-mse-loss)
- [`49` implement-adam-optimization-algorithm](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/49_implement-adam-optimization-algorithm)

### Exercise 3 — training a neuron end to end (DML #25)

Chain the sigmoid neuron from lesson 1 to the MSE gradients above: for `epochs`
rounds, do one full forward pass over the whole dataset, compute MSE, then take
one gradient-descent step on both `weights` and `bias`. Everything is rounded
to 4 decimals to match DML's own published outputs exactly.

In [ ]:
def train_neuron(features, labels, initial_weights, initial_bias, learning_rate, epochs):
    """DML #25: train a sigmoid neuron by full-batch gradient descent on MSE."""
    weights = np.asarray(initial_weights, dtype=float)
    bias = float(initial_bias)
    features = np.asarray(features, dtype=float)
    labels = np.asarray(labels, dtype=float)
    n = len(labels)
    mse_values = []

    for _ in range(epochs):
        # TODO(you): forward pass -- pre-activations, then sigmoid to get predictions
        z = ...
        preds = ...

        errors = preds - labels
        mse_values.append(round(float(np.mean(errors ** 2)), 4))

        # TODO(you): backprop through the sigmoid. d = dL/dz for each example is
        # errors * preds * (1 - preds) -- the MSE residual times the sigmoid
        # derivative preds*(1-preds). Then dw = (2/n) * X^T . d, db = (2/n) * mean(d)
        d = ...
        dw = ...
        db = ...

        weights = weights - learning_rate * dw
        bias = bias - learning_rate * db

    return np.round(weights, 4).tolist(), round(bias, 4), mse_values

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
w, b, mses = train_neuron(
    [[1.0, 2.0], [2.0, 1.0], [-1.0, -2.0]], [1, 0, 0], [0.1, -0.2], 0.0, 0.1, 2)
assert w == [0.1036, -0.1425] and b == -0.0167 and mses == [0.3033, 0.2942]

w, b, mses = train_neuron([[1, 2], [2, 3], [3, 1]], [1, 0, 1], [0.5, -0.2], 0, 0.1, 3)
assert w == [0.4892, -0.2301] and b == 0.0029 and mses == [0.21, 0.2087, 0.2076]

# Edge case: zero epochs is a no-op -- weights/bias pass through unchanged and
# no MSE is ever recorded.
w0, b0, mses0 = train_neuron([[1.0, 1.0]], [1], [0.5, 0.5], 0.1, 0.1, 0)
assert w0 == [0.5, 0.5] and b0 == 0.1 and mses0 == [], "epochs=0 -> untouched weights, empty history"

# MSE should trend downward across epochs on data the neuron can actually fit
_, _, mses_trend = train_neuron(
    [[2.0, 1.0], [-2.0, -1.0], [1.0, -1.0], [-1.0, 1.0]], [1, 0, 1, 0],
    [0.0, 0.0], 0.0, 0.5, 50)
assert mses_trend[-1] < mses_trend[0], "MSE should decrease over 50 epochs of gradient descent"
print("✅ DML 25 single-neuron-with-backpropagation passed")

<details>
<summary>💡 Show solution</summary>

```python
def train_neuron(features, labels, initial_weights, initial_bias, learning_rate, epochs):
    weights = np.asarray(initial_weights, dtype=float)
    bias = float(initial_bias)
    features = np.asarray(features, dtype=float)
    labels = np.asarray(labels, dtype=float)
    n = len(labels)
    mse_values = []
    for _ in range(epochs):
        z = features @ weights + bias
        preds = 1.0 / (1.0 + np.exp(-z))
        errors = preds - labels
        mse_values.append(round(float(np.mean(errors ** 2)), 4))
        d = errors * preds * (1 - preds)
        dw = 2 * features.T @ d / n
        db = 2 * np.mean(d)
        weights = weights - learning_rate * dw
        bias = bias - learning_rate * db
    return np.round(weights, 4).tolist(), round(bias, 4), mse_values
```

</details>

### Exercise 4 — a tiny scalar autograd engine (DML #26)

Every framework's `backward()` is the same idea at any scale: build a graph of
scalar operations, then walk it in reverse topological order accumulating
`grad` via the chain rule. Implement `+`, `*`, and `relu` for the `Value` class
below (this is the same design as Andrej Karpathy's
[micrograd](https://github.com/karpathy/micrograd)) -- each records its inputs
as `_prev` and stashes a `_backward` closure; `backward()` just calls all of
them in the right order.

In [ ]:
class Value:
    """DML #26: a scalar that remembers how it was computed, so it can run
    reverse-mode autodiff over +, *, and relu."""

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            # TODO(you): d(out)/d(self) = 1, d(out)/d(other) = 1 -- accumulate
            # out.grad into both (never overwrite; a Value can feed multiple ops)
            self.grad += ...
            other.grad += ...
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            # TODO(you): product rule -- d(out)/d(self) = other.data, d(out)/d(other) = self.data
            self.grad += ...
            other.grad += ...
        out._backward = _backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            # TODO(you): the local derivative is 1 where the output was positive, else 0
            self.grad += ...
        out._backward = _backward
        return out

    def backward(self):
        # Build a topological order (children before parents), then walk it
        # in reverse so every node's grad is fully accumulated before it
        # propagates further back.
        topo, visited = [], set()

        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)

        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()

In [ ]:
# Checks — run me (DML's own published test case, from tests.json)
a = Value(2); b = Value(3); c = Value(10)
d = a + b * c
e = Value(7) * Value(2)
f = e + d
g = f.relu()
g.backward()
assert (a.data, a.grad) == (2, 1)
assert (b.data, b.grad) == (3, 10)
assert (c.data, c.grad) == (10, 3)
assert (d.data, d.grad) == (32, 1)
assert (e.data, e.grad) == (14, 1)
assert (f.data, f.grad) == (46, 1)
assert (g.data, g.grad) == (46, 1)

# Edge case: relu clips a negative output to 0 AND stops the gradient flowing
# back through it (the classic "dead ReLU" -- ReLU's own local grad is 0 here).
x = Value(-3); y = Value(5)
z = (x + y * Value(-1)).relu()   # data = -3 + (5 * -1) = -8 -> ReLU clips to 0
z.backward()
assert z.data == 0 and x.grad == 0 and y.grad == 0, "a clipped ReLU stops gradient to everything upstream"

# Edge case: the same Value used twice in one expression must accumulate
# gradient from BOTH uses (not overwrite) -- this is why _backward uses +=.
p = Value(3)
q = p * p          # q = p^2, dq/dp = 2p = 6
q.backward()
assert q.data == 9 and p.grad == 6, "grad must accumulate across both uses of p in p*p"
print("✅ DML 26 basic-autograd-operations passed")

<details>
<summary>💡 Show solution</summary>

```python
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(0 if self.data < 0 else self.data, (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = 1
        for v in reversed(topo):
            v._backward()
```

</details>

### Exercise 5 — batch, stochastic, and mini-batch gradient descent (DML #47)

One function, three variants, all on plain MSE. The key gotcha (confirmed
against DML's own test cases): `n_iterations` means **full passes over the
dataset (epochs)** for `'stochastic'` and `'mini_batch'` -- each epoch visits
every sample once, in original order (no shuffling) -- while `'batch'` always
uses the whole dataset, so one iteration there already is one epoch.

In [ ]:
def gradient_descent(X, y, weights, learning_rate, n_iterations, batch_size=1, method='batch'):
    """DML #47: batch / stochastic / mini-batch gradient descent on MSE, no shuffling."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    m = len(y)

    for _ in range(n_iterations):
        if method == 'batch':
            batches = [(X, y)]
        elif method == 'stochastic':
            # TODO(you): one epoch = every sample, one at a time, in order
            batches = ...
        elif method == 'mini_batch':
            # TODO(you): one epoch = consecutive chunks of `batch_size`, in order
            batches = ...
        else:
            raise ValueError(f"unknown method: {method}")

        for X_b, y_b in batches:
            # TODO(you): MSE gradient for this (sub)batch: 2/n_b * X_b^T . (X_b @ w - y_b)
            errors = ...
            grad = ...
            weights = weights - learning_rate * grad

    return weights

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
X = np.array([[1, 1], [2, 1], [3, 1], [4, 1]])
y = np.array([2, 3, 4, 5])
w0 = np.zeros(X.shape[1])

out_batch = gradient_descent(X, y, w0, 0.01, 100, method='batch')
assert np.allclose(out_batch, [1.14905239, 0.56176776], atol=1e-6)

out_sgd = gradient_descent(X, y, w0, 0.01, 100, method='stochastic')
assert np.allclose(out_sgd, [1.0507814, 0.83659454], atol=1e-6)

out_mb = gradient_descent(X, y, w0, 0.01, 100, batch_size=2, method='mini_batch')
assert np.allclose(out_mb, [1.10334065, 0.68329431], atol=1e-6)

# Edge case: batch_size >= len(dataset) collapses mini-batch onto plain batch
# gradient descent (a single "chunk" covering everything, every epoch).
out_mb_full = gradient_descent(X, y, w0, 0.01, 100, batch_size=len(y), method='mini_batch')
assert np.allclose(out_mb_full, out_batch, atol=1e-6), "batch_size = n -> mini-batch reduces to batch GD"

# Edge case: n_iterations=0 -- weights must pass through completely untouched
assert np.allclose(gradient_descent(X, y, w0, 0.01, 0, method='batch'), w0)
print("✅ DML 47 gradient-descent-variants passed")

<details>
<summary>💡 Show solution</summary>

```python
def gradient_descent(X, y, weights, learning_rate, n_iterations, batch_size=1, method='batch'):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    weights = np.asarray(weights, dtype=float)
    m = len(y)
    for _ in range(n_iterations):
        if method == 'batch':
            batches = [(X, y)]
        elif method == 'stochastic':
            batches = [(X[i:i+1], y[i:i+1]) for i in range(m)]
        elif method == 'mini_batch':
            batches = [(X[i:i+batch_size], y[i:i+batch_size]) for i in range(0, m, batch_size)]
        else:
            raise ValueError(f"unknown method: {method}")
        for X_b, y_b in batches:
            errors = X_b @ weights - y_b
            grad = 2 * X_b.T @ errors / len(y_b)
            weights = weights - learning_rate * grad
    return weights
```

</details>

### Exercise 6 — Adam in its "optimize this function" signature (DML #49)

`adam_step` above updates one parameter vector per call, with the caller
managing `state` between steps -- the shape you'd want inside a training loop.
DML #49 asks for the same algorithm wrapped the other way around: hand Adam
the objective `f`, its `grad`, and a starting point `x0`, and let it run
`num_iterations` steps internally. Same math (first/second moment EMAs with
bias correction), different ergonomics.

In [ ]:
def adam_optimizer(f, grad, x0, learning_rate=0.001, beta1=0.9, beta2=0.999,
                   epsilon=1e-8, num_iterations=10):
    """DML #49: Adam, self-contained -- runs its own loop instead of exposing state."""
    x = np.asarray(x0, dtype=float)
    m = np.zeros_like(x)
    v = np.zeros_like(x)

    for t in range(1, num_iterations + 1):
        g = grad(x)
        # TODO(you): update the biased first/second moment EMAs
        m = ...
        v = ...
        # TODO(you): bias-corrected estimates (t starts at 1, so beta**t < 1 early on)
        m_hat = ...
        v_hat = ...
        x = x - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)

    return x

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
def objective(x):
    return x[0] ** 2 + x[1] ** 2

def grad_fn(x):
    return np.array([2 * x[0], 2 * x[1]])

out1 = adam_optimizer(objective, grad_fn, np.array([1.0, 1.0]))
assert np.allclose(out1, [0.99000325, 0.99000325], atol=1e-6)

out2 = adam_optimizer(objective, grad_fn, np.array([0.2, 12.3]))
assert np.allclose(out2, [0.19001678, 12.29000026], atol=1e-6)

# Edge case: num_iterations=0 -- x0 passes straight through unchanged
assert np.allclose(adam_optimizer(objective, grad_fn, np.array([5.0, -5.0]), num_iterations=0), [5.0, -5.0])

# Sanity: however it's parameterized, Adam should still descend towards the
# minimum of a convex bowl from any starting point (each step moves by
# roughly `learning_rate` regardless of gradient size, so with the default
# lr=0.001 this needs many steps -- check direction of progress, not distance).
out3 = adam_optimizer(objective, grad_fn, np.array([50.0, -30.0]), num_iterations=2000)
assert np.all(np.abs(out3) < np.abs(np.array([50.0, -30.0]))), \
    "should have made real progress toward (0, 0), not moved away from it"
print("✅ DML 49 adam-optimization-algorithm passed")

<details>
<summary>💡 Show solution</summary>

```python
def adam_optimizer(f, grad, x0, learning_rate=0.001, beta1=0.9, beta2=0.999,
                   epsilon=1e-8, num_iterations=10):
    x = np.asarray(x0, dtype=float)
    m = np.zeros_like(x)
    v = np.zeros_like(x)
    for t in range(1, num_iterations + 1):
        g = grad(x)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        x = x - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
    return x
```

</details>